# BoxBox

BoxBox explains a pit-stop call from a historical race card. It answers BOX, STAY OUT or NO CALL with one sentence of justification, and it gives no call when the card does not support one.

## 1. Installation and environment

We record the library versions, the device and the pinned model revision, so that every number below can be reproduced. The install commands are in `README.txt`.

In [ ]:
# Versions, device and model identity
import importlib.metadata as metadata
import platform

from huggingface_hub import model_info

from boxbox.config import RunConfig
from boxbox.model import get_device

config = RunConfig()
for package in ("torch", "transformers", "peft", "trl", "accelerate", "datasets"):
    print(f"{package:<13}{metadata.version(package)}")
print(f"python       {platform.python_version()} on {platform.machine()}")
print(f"device       {get_device()}")
print(f"model        {config.model_id} @ {config.model_revision[:12]}")
print(f"licence      {model_info(config.model_id, revision=config.model_revision).card_data.license}")

The notebook runs on MPS with torch 2.14.1 and transformers 5.20.1. The model is pinned to revision a07cc9a04f16 and is released under the Apache 2.0 licence.

## 2. Observe the model

We load SmolLM3-3B in bf16 and cap the accelerator memory at 20 GB, the budget agreed for training. Its shape tells us how wide each token representation is and how attention is split.

In [ ]:
# Load the model and print its dimensions
from boxbox.model import cap_mps_memory, load_model, memory_gb

cap_mps_memory(20)
model, tokenizer = load_model(config)
shape = model.config
print(f"parameters   {sum(p.numel() for p in model.parameters()) / 1e9:.2f} B")
print(f"layers       {shape.num_hidden_layers}")
print(f"hidden size  {shape.hidden_size}")
print(f"heads        {shape.num_attention_heads} query, {shape.num_key_value_heads} key/value")
print(f"vocabulary   {shape.vocab_size}")
print(f"no RoPE in   {shape.no_rope_layers.count(0)} of {shape.num_hidden_layers} layers")
print(f"tied embeddings {shape.tie_word_embeddings}")
print(f"memory       {memory_gb(get_device()):.1f} GB")

SmolLM3-3B has 3.08 B parameters in 36 layers with a hidden size of 2048. Its 16 query heads share 4 key/value heads, which is grouped-query attention, and 9 of the 36 layers have no rotary position encoding. The model takes 6.5 GB in bf16, which leaves room for training under the 20 GB cap.

### 2.1 Token IDs

We tokenize one line in the style of a race card. Numbers, car numbers and the CALL label are what the model must copy exactly, so we look at how they split.

In [ ]:
# Token IDs of a race-card line
text = "Car 44, tyres aged 18 laps, gap behind 24.0 s, pit loss 21.0 s. CALL: BOX"
ids = tokenizer(text)["input_ids"]
print(ids)
print([tokenizer.decode([token]) for token in ids])
print(f"{len(text)} characters, {len(ids)} tokens")

The 73 characters give 29 tokens. Numbers are split into digit groups with the space as its own token, so "24.0" becomes `' '`, `'24'`, `'.'`, `'0'`. The car number `'44'` and the label `' BOX'` are single tokens, so the call after "CALL:" is decided in one next-token step.

### 2.2 Logits shape

We build the chat prompt for the `box_old` fixture card with the model's own template, in no-think mode. We add the start of the answer, `CALL:`, so the next token is the call itself.

In [ ]:
# Chat prompt for the box_old card, then one forward pass
import json
from pathlib import Path

import torch

card = json.loads(Path("reference/fixtures.json").read_text())["conversations"][0]["card"]
messages = [
    {"role": "system", "content": "You are BoxBox. Answer from the race card only. /no_think"},
    {"role": "user", "content": f"{json.dumps(card)}\n\nShould car 44 box now?"},
]
prompt = tokenizer.apply_chat_template(messages, add_generation_prompt=True, tokenize=False)
input_ids = tokenizer(prompt + "CALL:", return_tensors="pt")["input_ids"].to(model.device)
with torch.no_grad():
    logits = model(input_ids).logits
print(prompt[-120:])
print(f"input_ids {tuple(input_ids.shape)}, logits {tuple(logits.shape)}")

In no-think mode the template inserts an empty `<think>` block, and the answer starts right after it. The prompt has 230 tokens and the logits have shape (1, 230, 128256): one score per vocabulary token at every position. Only the last row predicts the next token.

### 2.3 Next-token distribution

The softmax of the last row of logits gives the probability of every token in the vocabulary. We read the ten most likely tokens and the first token of each call.

In [ ]:
# Softmax over the vocabulary after "CALL:"
from boxbox.model import next_token_probs, top_tokens

probs = next_token_probs(model, input_ids)
print(f"sum of probabilities {probs.sum().item():.4f}")
for text_, token, p in top_tokens(probs, tokenizer, 10):
    print(f"{token:>7} {text_!r:<12} {p:.4f}")
for label in (" BOX", " STAY", " NO"):
    first = tokenizer(label, add_special_tokens=False)["input_ids"][0]
    print(f"P({label.strip()}) = {probs[first].item():.4f}")

The probabilities sum to 1. After "CALL:" the base model prefers a space (0.1822) or a quote (0.1105) to `' BOX'` (0.0975). Among the three calls, BOX is clearly ahead of STAY (0.0005) and NO (0.0080), which matches the rule's answer on this card. The format itself is not yet the model's first choice.

### 2.4 Argmax, temperature, top-k and top-p

Argmax always keeps the most likely token. Temperature divides the logits before the softmax, so it sharpens or flattens the distribution. We measure that effect, then sample whole answers.

In [ ]:
# Argmax and the effect of temperature on the same logits
top = probs.argmax().item()
print(f"argmax: {top} {tokenizer.decode([top])!r}")
for temperature in (0.5, 1.0, 1.5):
    p = next_token_probs(model, input_ids, temperature)
    entropy = -(p * p.clamp_min(1e-12).log()).sum().item()
    best = [(t, round(v, 3)) for t, _, v in top_tokens(p, tokenizer, 3)]
    print(f"T={temperature}: entropy {entropy:.3f} nats, top 3 {best}")

Argmax keeps the space token at every temperature, because dividing the logits by T does not change their order. T = 0.5 sharpens the distribution (entropy 1.291 nats, the space at 0.568), and T = 1.5 flattens it (8.961 nats, the top token at 0.028).

In [ ]:
# Whole answers: greedy, then top-k 20 and top-p 0.9 samples at temperature 1
prompt_ids = tokenizer(prompt, return_tensors="pt")["input_ids"].to(model.device)
settings = {
    "greedy": {"do_sample": False},
    "top-k 20": {"do_sample": True, "top_k": 20, "top_p": 1.0, "temperature": 1.0},
    "top-p 0.9": {"do_sample": True, "top_k": 0, "top_p": 0.9, "temperature": 1.0},
}
for name, options in settings.items():
    for seed in (1, 2) if options["do_sample"] else (0,):
        torch.manual_seed(seed)
        out = model.generate(prompt_ids, max_new_tokens=48, **options)
        answer = tokenizer.decode(out[0, prompt_ids.shape[1]:], skip_special_tokens=True)
        print(f"[{name}, seed {seed}] {answer.strip()!r}")

The greedy answer ignores the two-line format and gets the comparison wrong: it says that 24 seconds is less than the 21-second pit loss. A sampled answer adds a claim the card does not make, that car 44 "has not yet encountered the pit loss". With seed 2, top-k 20 and top-p 0.9 produced the same text, since both keep the same few likely tokens at each step here. Greedy decoding stays the safer choice for evaluation, and the format and the arithmetic become targets for prompting and LoRA.

### 2.5 Embeddings, causal attention and unembedding

Each token ID selects one row of the embedding matrix. With tied embeddings, the same matrix turns the last hidden state back into logits. We check that the logits are exactly this product.

In [ ]:
# Embedding matrix, tied weights and logits = last hidden state x embeddings
embeddings = model.get_input_embeddings().weight
print(f"embedding matrix {tuple(embeddings.shape)}")
print(f"output layer shares it: {model.get_output_embeddings().weight is embeddings}")
with torch.no_grad():
    out = model(input_ids, output_hidden_states=True)
hidden = out.hidden_states[-1][0, -1]
manual = hidden @ embeddings.T
gap = (manual.float() - out.logits[0, -1].float()).abs().max().item()
print(f"last hidden state {tuple(hidden.shape)}, largest gap to the logits {gap:.4f}")

The output layer uses the embedding matrix itself, of shape 128256 × 2048. Multiplying the last hidden state by its transpose gives the logits within 0.0625, which is bf16 rounding. The score of `' BOX'` is therefore the dot product between the final hidden state and the embedding of `' BOX'`.

Attention is causal: a token can only attend to itself and earlier tokens. We switch to the eager attention kernel to read the weights of the first layer on the short race-card line.

In [ ]:
# Attention weights of layer 1, head 1, on the race-card line
import matplotlib.pyplot as plt

short = tokenizer(text, return_tensors="pt")["input_ids"].to(model.device)
model.set_attn_implementation("eager")
with torch.no_grad():
    weights = model(short, output_attentions=True).attentions[0][0, 0].float().cpu()
model.set_attn_implementation("sdpa")
print(f"weights {tuple(weights.shape)}, rows sum to {weights.sum(-1).mean():.3f}")
print(f"weight above the diagonal: {weights.triu(1).sum().item():.6f}")
labels = [tokenizer.decode([t]) for t in short[0]]
plt.figure(figsize=(7, 6))
plt.imshow(weights, cmap="Blues")
plt.xticks(range(len(labels)), labels, rotation=90, fontsize=7)
plt.yticks(range(len(labels)), labels, fontsize=7)
plt.title("Layer 1, head 1: each row attends only to earlier tokens")
plt.tight_layout()
plt.show()

Each row of the 29 × 29 matrix sums to 1, and the weight above the diagonal is exactly 0. The causal mask lets each token attend only to itself and earlier tokens, which is what allows generation one token at a time.